# 13 — Cam-CAN → HCP: train on one cohort, test on the other

The model is fitted on **every** Cam-CAN subject and applied once to the HCP subjects it has never
seen. One number per configuration, scored against HCP's own questionnaire.

**Why not more cross-validation.** `select-bstm`'s five folds each estimate as many coefficients as
the whole model has, which is what makes 168 subjects a poor *training* cohort and a decent *test*
set. Simulated with this pipeline's own estimator, the same 168 subjects give about **11%** power to
detect a true R² of 0.20 as a training cohort, and about **83%** power to detect ρ = 0.20 as a test
set. Splitting training from testing uses each cohort for what it can do.

**No equating of the two questionnaires is needed.** Spearman uses ranks. The prediction comes out
in Cam-CAN's units and the truth is in the ASR's, and the two are never compared as values — only as
order. "Does the model rank these people the way their own instrument does" is a far weaker
assumption than making HADS and the ASR commensurate, which nothing here could support: the
instruments share no items and no person took both.

**What a null here would and would not mean.** Three things must hold for a score to appear — the
state space must carry over from 3T to 7T, the model from one age range to another, and the two
questionnaires must measure the same thing in the same direction. A zero does not say which failed.
So **the within-Cam-CAN result stays the primary finding and this is confirmation**; if Cam-CAN's own
CV is null, a null here adds nothing.

---

| section | question |
|---|---|
| **1** | are the two cohorts comparable at all? *(precondition — before any score)* |
| **2** | the whole multiverse, transferred |
| **3** | one model in depth |

## 0. Setup

Everything about the two cohorts — the task maps, the target columns, the covariate spelling, the
pooling — lives in **`notebooks/transfer_config.py`**, because two callers need it: this notebook
draws the figures and `tools/transfer_grid.py` computes the rows on the cluster. If each built its
own `Settings`, the figures would eventually be drawn from rows produced under a different target
range or a different covariate map, and nothing in the figure would say so.

**Both targets are run**: the ordinal category the report has always used, and the raw score. The
raw has 22 levels against the category's 4, so it gives the model far more to rank — at the cost of
not being what the existing results used. They are written under separate `targets=` partitions and
are never concatenated.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(REPO / "notebooks"), str(REPO)]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import hads_tables as ht, hads_plots as hp
import cohort_plots as cp, cohort_tables as ct
import transfer_tables as tt, transfer_config as tcfg

SAVE_DIR = None          # e.g. REPO / "notebooks" / "figures" / "13"

# Which Cam-CAN target set. The rows for BOTH are written by the cluster job;
# this only chooses which to read and draw.
TARGET = "raw"           # "raw" | "category"

# Where tools/transfer_grid.py wrote them.
TRANSFERRED = None       # None = <outputs>/transferred_models

In [ ]:
# One list, filled by the pooling so the next cell can report it once instead
# of 564 times.
POOLED = []

train, test, train_cfg, test_cfg = tcfg.studies(
    TARGET, save_dir=SAVE_DIR, pool_record=POOLED)
ROOT = Path(TRANSFERRED) if TRANSFERRED else train_cfg.root / "transferred_models"

for name, st in (("train", train), ("test", test)):
    p = st.pheno
    print(f"{name:5s}  {len(p):,} subjects;  "
          f"Age {p['Age'].min():.0f}-{p['Age'].max():.0f};  "
          f"Sex {dict(p['Sex'].value_counts())};  "
          + ", ".join(f"{lab} n={int(p[lab].notna().sum())}" for lab in st.cfg.targets))
print(f"\ntargets  train {dict(train_cfg.targets)}")
print(f"         test  {dict(test_cfg.targets)}")
print(f"rows     {ROOT}")

### Pooling HCP's four runs

HCP gives MOVIE1–4 and REST1–4 per subject; the models score **one row per subject** and will not
average them. Averaging the columns would be wrong for most: an entropy rate averaged over runs is
not the entropy rate of anything, a switch rate is a ratio whose denominators differ, and occupancy
is a fraction of a different number of windows each time.

`pool_tasks` recomputes instead, and `transfer_config` wires it into `load_set` as `pool_fn` — so
the pooling happens before any arm is built and before the phenotype join, and every arm of one cell
sees the same pooled row. `p_joint × n_transitions` is the count matrix, and counts add, so the
cells, `n_transitions` and occupancy pool **exactly**; the rates and the entropy are then recomputed
from the pooled matrix. The one approximation is `dispersion`, which needs the latents rather than
this table — it is flagged in the frame.

What pooling *does* leave undefined: `mean_dwell_s` and `switches_per_min` need the stride, recovered
per run from `switches_per_min`, which a run where nobody switched gives away as 0/0. Those NaNs are
filled with the **training** column mean before any fit, and the count is reported in §2.

In [ ]:
# Verify the pooling once, on one cell, rather than trusting it in silence.
shared = tt.common_sets(train, test)
print(f"{len(shared)} state set(s) both cohorts have a transition table for")
probe = ht.space_of(shared.iloc[0])

for task in test_cfg.tasks:
    t = test.load_set(probe, task)
    n_in, n_out = POOLED[-1] if POOLED else (len(t), len(t))
    runs = t["n_runs_pooled"].max() if "n_runs_pooled" in t else 1
    print(f"  {task:5s}  {n_in:4d} run-row(s) -> {n_out:4d} subject-row(s), "
          f"up to {runs:.0f} run(s) per subject;  task reads "
          f"{t['task'].iloc[0]!r}")
    assert t["sub"].is_unique, "pooling left a subject twice"

# and the training side, which is one run per condition and must NOT pool
for task in train_cfg.tasks:
    t = train.load_set(probe, task)
    print(f"  {task:5s}  {len(t):4d} Cam-CAN row(s);  task reads "
          f"{t['task'].iloc[0]!r}")

## 1. Are the two cohorts comparable at all?

**A precondition, not a result.** If HCP's brain states do not behave like Cam-CAN's, a model
trained on one cannot transfer to the other, and the score below would be measuring that rather than
anything about symptoms. These are notebook 11's figures with HCP in place of the other cohorts.

In [ ]:
# cohort_tables has its OWN Settings, keyed by one state set and a cohort->dirs
# map, not hads_tables'. Pick the space section 2 ends up caring about; this
# one is a precondition check, so a representative set is enough.
# THE ONE KNOB for this section: `space`. `shared` above lists every set the
# two cohorts have, so to check another one, change these three values. Worth
# re-running on whatever set section 3 ends up choosing -- a model leaning on a
# transition whose cohorts are NOT equivalent is a model whose transfer was
# never going to work.
co_cfg = ct.Settings(
    space=dict(atlas="yeo7", window_s=-1.0, states="HMM1_pca3_8"),
    cohorts={"ds002837":  dict(dirs=["ds002837"],  role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan":    dict(dirs=["camcan", "camcan_rest"], role="projected"),
             "hcp7t_fix": dict(dirs=["hcp7t_fix"], role="projected")},
    save_dir=SAVE_DIR,
)
co = ct.Cohorts(co_cfg)
agree = co.agreement()
cp.fig_tost(co, agree)          # 2a: is every cohort equivalent to the others?

In [ ]:
groups = co.groups("cohort")
cp.fig_matrices(co, groups, "tost", title="4a  transitions by cohort",
                name="13_4a_matrix_cohort")
cp.fig_matrices(co, groups, "cohen", sd=True, title="4b  transitions by cohort",
                name="13_4b_matrix_cohort")

## 2. The whole multiverse, transferred

Every configuration the two cohorts share: fit on all of Cam-CAN, one score on HCP.

**The fits happen on the cluster, not here.** One fit per (state set × task × target × arm × model)
on *every* training subject is ~9,000 fits plus ~2,250 baselines over 282 state sets — hours in a
kernel that cannot be resumed if the connection drops, minutes as an array. So:

```bash
./slurm/transfer_grid.sh                 # both target sets, 40 shards + the fcm job
./slurm/transfer_grid.sh -- --atlas yeo7 # a slice first, to see the chain work
```

and this notebook reads the parquet. The reader **refuses a half-finished grid**: a smaller
multiverse reads as a valid one, and "the best of 282 state sets" from the shards that happened to
finish is a different claim in the same words.

**The error bar means something different here.** In the Cam-CAN report `± sd` is the spread over
fold seeds — how much the answer moves when the folds are redrawn. There are no folds now. It is the
bootstrap sd over the **test subjects**: how much the answer moves when a different 168 people are
tested. Both are error bars; they answer different questions.

The baseline is Age + Sex trained on Cam-CAN and applied to HCP the same way, so the comparison is
like for like.

In [ ]:
raw_rows = tt.read_transferred(ROOT, TARGET, trees=("bstm", "fcm"))
# `drop_degenerate` removes the rows where the model predicted ONE value for
# every test subject -- Spearman has nothing to rank there, so the score is
# NaN. They stay on disk because they are an outcome; it prints which (model,
# arm) they happened to, because "hgb predicted a constant everywhere" is a
# finding about hgb and not a plotting detail.
rows = tt.drop_degenerate(tt.as_report_rows(raw_rows))
prov = raw_rows.iloc[0]
print(f"{len(rows)} configuration(s) over {rows['states'].nunique()} state set(s);  "
      f"{rows['over_base'].mean():.0%} above their transferred baseline")
print(f"  produced by {prov['git_sha']} on {prov['written_at']}, "
      f"seed {prov['seed']}, {prov['n_boot']:,} bootstrap draws, "
      f"grid {prov['grid_hash']}")
if raw_rows["n_imputed_test"].sum():
    print(f"  {int(raw_rows['n_imputed_test'].sum()):,} test value(s) filled with a "
          f"TRAINING column mean (pooling leaves some dynamics undefined); "
          f"{int(raw_rows['n_columns_dropped'].max())} column(s) dropped at most")
# WHAT WAS CONCEDED. A frame mismatch allowed with --allow-frame-mismatch is
# recorded on the row, not just in the job log, because a concession that
# travels only in a log is one nobody reads again.
_m = raw_rows["frames_mismatch"] if "frames_mismatch" in raw_rows else None
if _m is not None and _m.fillna("").ne("").any():
    _bad = raw_rows[_m.fillna("").ne("")]
    print(f"\n!! {len(_bad)} fcm row(s) were scored over DIFFERENT frames:")
    for why, n in _bad["frames_mismatch"].value_counts().items():
        print(f"   {n:>4} row(s)  {why}")
    print("   Allowed on purpose. It has to be stated wherever these rows "
          "appear: in them, a rest-vs-movie or train-vs-test difference is "
          "PARTLY a difference in preprocessing, not in condition or cohort.")

rows.sort_values("score", ascending=False).head(12)[
    ["atlas", "window", "states", "task", "phenotype", "arm", "model",
     "score", "lo", "hi", "base_score"]]

### The fcm half

Static FC, transferred the same way, so the four tables can be compared. It came out of the same
job (`TREE=fcm`, one task — static FC has no aperture and no state set), and is already in `rows`.

Two decisions had to come from the **training** cohort alone, and both would be invisible if they
went wrong:

- **which edges.** `fcm_selection` drops an edge that is NaN in more than `max_missing` of subjects,
  so the surviving set follows each cohort's own coverage. Let HCP choose its own and the model is
  handed a different feature space than it was fitted on — with the columns still lining up by count.
- **the imputed value.** A NaN is filled with a column mean. Filled from HCP's mean, every test
  subject's missing edge is nudged toward HCP's centre: adaptation to the data being scored.

Pooling HCP's four runs is **not exact here**, unlike on the transition side. A correlation of four
concatenated runs is not a function of the four correlations — each was computed against its own
run's mean. What is done is the standard meta-analytic pooling: a mean of Fisher-z values weighted
by `n_tr_used`. The exact alternative would be `static-fc --pool` taking a task filter, which does
not exist.

In [ ]:
fcm = rows[rows["tree"] == "fcm"]
print(f"{len(fcm)} fcm configuration(s);  "
      f"{fcm['over_base'].mean():.0%} above their transferred baseline"
      if len(fcm) else "no static-FC rows: was the TREE=fcm job submitted?")
_cols = ["atlas", "task", "phenotype", "arm", "model", "score", "lo", "hi",
         "base_score", "n_edges_used", "n_test"]
# the frames each side was correlated over, beside the score, so the table
# cannot be read without them
_cols += [c for c in ("train_censor_policy", "test_censor_policy")
          if c in fcm.columns]
display(fcm[_cols])

In [ ]:
# `fallback=train.above_zero`: where NOTHING clears the Age + Sex baseline at
# both conditions, colour what is at least above ZERO at both instead. The two
# are different claims -- better than chance, and better than Age and Sex --
# so the panel's own caption says which one it is showing. With HCP's anxiety
# baseline at rho ~ 0.2, that distinction is the result.
hp.fig_rankings(rows, train_cfg, train.above_both,
                fallback=train.above_zero)      # B1, transferred

In [ ]:
hp.fig_factors(rows, train_cfg)                         # B2, transferred

In [ ]:
hp.fig_k_curve(rows, train_cfg)                         # B2.K, transferred

## 3. One model in depth

Pick the configuration to look at. The default is the best transferred score, but a row that only
just cleared its baseline on a lucky bootstrap is not a finding — prefer one whose **interval**
clears it.

In [ ]:
# WHICH MODEL SECTION 3 LOOKS AT. None = the best row of any model.
#
# Set it to "ridge" to draw E1's SHAP panel in an image WITHOUT the `shap`
# package: ridge's SHAP is exact arithmetic -- coefficient x standardised
# value -- and hads_tables computes it with numpy alone. A boosted tree's
# needs shap.TreeExplainer, and containers/stage45_shap.def is the image that
# has it. The panel is then a statement about the ridge model, which is worth
# saying out loud wherever it appears.
E1_MODEL = None          # None | "ridge" | "hgb"

pool = rows if E1_MODEL is None else rows[rows["model"] == E1_MODEL]
if not len(pool):
    raise SystemExit(f"no row with model={E1_MODEL!r}; rows hold "
                     f"{sorted(rows['model'].unique())}")
clears = pool[pool["lo"] > pool["base_score"]].sort_values("score", ascending=False)
print(f"{len(clears)} of {len(pool)} configuration(s) have an interval clearing their baseline")
best = (clears if len(clears) else pool.sort_values("score", ascending=False)).iloc[0]
space = {"atlas": best["atlas"], "window_s": best["window_s"],
         "states": best["states"]}
print(f"chosen: {best['label']}  ·  {best['arm']}  ·  {best['model']}")
print(f"  score {best['score']:+.3f}  [{best['lo']:+.3f}, {best['hi']:+.3f}]"
      f"   baseline {best['base_score']:+.3f}")
if not len(clears):
    print("  !! nothing clears its baseline -- this row is the best of a null set")

### E1 — how the chosen set scores, bstm beside fcm, and what it uses

Notebook 10's E1, with two differences that the transfer forces.

**One point per (target, condition, tree).** The fcm half has two arms — `edges`, the hypothesis,
and `global`, its control (a subject's mean and SD over every edge, two numbers with no topology in
them). `train_cfg.fc_arm` picks which one stands beside the state model here; `global` belongs in a
separate comparison, not sharing an axis label with `edges`.

**The SHAP is on the test subjects.** In notebook 10 it is out-of-fold inside Cam-CAN: what the
model used while learning. There are no folds here — one fit on all 649 — so the question that can
be asked is the better one: what it used on the 168 subjects it was *scored* on. Same blocks, same
normalisation, different statement.

In [ ]:
perf = tt.transfer_perf(rows, best, fc_arm=train_cfg.fc_arm)
display(perf[["tree", "arm", "task", "phenotype", "score", "lo", "hi",
              "base_score", "n_test"]])

# |SHAP| of the Cam-CAN-fitted model, evaluated on the HCP rows it scored
shares = tt.transfer_shares(train, test, space, best["arm"], best["model"])

hp.fig_performance(
    perf, train_cfg,
    title=f"E1  transferred · {best['label']} · {best['model']}",
    legend={"bstm": f"{best['label']} · {best['arm']}",
            "fcm": f"{best['atlas']} · {train_cfg.fc_arm}"},
    model=best["model"], shares=shares,
    share_label="share of mean |SHAP| on the TEST subjects "
                "(one fit on all of Cam-CAN, no folds)")

### E3 — where the model looks

SHAP of the model **trained on Cam-CAN**, which is notebook 10's E3 unchanged: nothing about that
model differs here, only where it is applied. Rather than duplicate notebook 10's setup — `deep`,
`model_marks`, the reference and colour maps — open **notebook 10 section E3** with
`SPACE` set to the space printed above.

What this notebook adds is the reason to look: read E3 beside section 1. A model leaning on a
transition whose two cohorts are **not** equivalent is a model whose transfer was never going to
work, and that is visible by putting the two figures side by side.

In [ ]:
print("open notebook 10 at E3 with:")
print(f'  SPACE = dict(atlas="{best["atlas"]}", '
      f'window_s={best["window_s"]}, states="{best["states"]}")')
print(f'  E1_MODEL = "{best["model"]}"     # arm: {best["arm"]}')

## What to write down

1. **Section 1 first.** If the cohorts are not equivalent, sections 2 and 3 are measuring the
   mismatch.
2. **An interval that clears the baseline** is the claim. A point estimate above it with an interval
   that crosses is 168 subjects, not a finding.
3. **A null is ambiguous** — state space, model, or instrument, and this cannot separate them. Say
   so rather than concluding the transitions do not predict.
4. Re-run with `TARGET = "raw"` and `TARGET = "category"` and compare — the cluster job writes both,
   so this is a re-read, not a refit. The raw has 22 levels against 4, so it should rank **better**
   if the signal is real and worse if the category was doing the work.
5. **Whether the four arms share their frames.** They do, here: Cam-CAN is **one** cohort with
   two conditions (`cohort=camcan`, `task=Movie` and `task=Rest`), so both arms were built by one
   run of each stage, under `censor_policy=motion` — the same policy HCP's tables carry. §2 prints
   any row where that stops being true, and the fcm transfer refuses a mismatch unless
   `--allow-frame-mismatch` is passed, which records it on every row. Check that §2 prints no such
   rows before reading a rest-vs-movie or a bstm-vs-fcm difference as a difference in condition or
   model.
6. **What was imputed.** Pooling leaves `mean_dwell_s` and `switches_per_min` undefined for a subject
   whose runs record no switch, and `dispersion` needs the latents rather than the transition table.
   Those are filled with the **training** cohort's column mean, never HCP's, and §2 prints how many.
   If that count is large relative to 168 × the number of dynamics columns, the `dynamics` arm is
   partly measuring the imputation and should be said so.